# Panda robot reach: DDPG + HER (updated tutorial notebook)

The YouTube tutorial trained DDPG + HER on panda-gym's `PandaReach-v2` with panda-gym 2.0.0, gym, Stable-Baselines3 1.5
and Python 3.7. This version runs top to bottom on today's libraries (panda-gym 3.x / gymnasium, SB3 2.x,
sb3-contrib, CPU torch) with **the same hyperparameters**.

Changes from the original:
- `PandaReach-v2` -> `PandaReach-v3` (panda-gym 3.x uses `gymnasium`, not `gym`; `reset()` returns `(obs, info)`,
  `step()` returns `(obs, reward, terminated, truncated, info)`)
- `online_sampling=True` dropped from the HER kwargs: SB3 >= 1.8 always samples HER goals online
- `max_episode_length` is no longer passed to `HerReplayBuffer` (read from the env)
- SB3 2.x DDPG trains 1 gradient step per env step (`train_freq=1, gradient_steps=1`); SB3 1.5 ran 50 gradient
  steps after each 50-step episode. Same update-to-data ratio.
- Training on CPU: ~3 env steps/s on one core with these settings (the tutorial ran ~48 steps/s on a GPU),
  so `TOTAL_STEPS` defaults to 100k instead of 1e6. The success rate plateaus long before 100k.


In [ ]:
# !pip install "panda-gym==3.0.7" "stable-baselines3==2.9.2" "sb3-contrib==2.9.2" imageio imageio-ffmpeg

In [ ]:
import os
import gymnasium as gym
import numpy as np
import panda_gym  # registers the Panda envs
import torch
from sb3_contrib.common.wrappers import TimeFeatureWrapper
from stable_baselines3 import DDPG, HerReplayBuffer
from stable_baselines3.common.noise import NormalActionNoise

TOTAL_STEPS = int(os.environ.get("TOTAL_STEPS", 100_000))  # tutorial: 1e6
torch.set_num_threads(os.cpu_count())

## Environment
Sparse reward (-1 until the end-effector is within 5 cm of the goal, then 0), 50-step episodes. `TimeFeatureWrapper` appends the remaining-time fraction to the observation.

In [ ]:
env = TimeFeatureWrapper(gym.make("PandaReach-v3"))
obs, info = env.reset(seed=0)
print(env.observation_space)
print(env.action_space)

## Model (the tutorial's settings)
`n_critics=2` makes SB3's DDPG keep two Q-networks and use the minimum of the two as the target (clipped double-Q, one of TD3's three tricks), so this is not textbook DDPG. Set `n_critics=1` for pure DDPG.

In [ ]:
n_actions = env.action_space.shape[0]
model = DDPG(
    "MultiInputPolicy", env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(goal_selection_strategy="future", n_sampled_goal=4),
    gamma=0.95, batch_size=2048, buffer_size=100_000, learning_rate=1e-3,
    action_noise=NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions)),
    policy_kwargs=dict(net_arch=[512, 512, 512], n_critics=2),
    verbose=1, seed=0,
)

In [ ]:
model.learn(total_timesteps=TOTAL_STEPS, log_interval=20)  # prints rollout/success_rate
model.save("ddpg_her_panda_reach")

## Test: 100 deterministic episodes

In [ ]:
successes, final_dist = [], []
for i in range(100):
    obs, info = env.reset(seed=2_000_000 + i)
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
    successes.append(info["is_success"])
    final_dist.append(np.linalg.norm(obs["achieved_goal"] - obs["desired_goal"]))
print(f"success rate {np.mean(successes):.0%}, mean final distance {100 * np.mean(final_dist):.1f} cm")

## Video (headless, `render_mode="rgb_array"`)

In [ ]:
import imageio.v2 as imageio

venv = TimeFeatureWrapper(gym.make("PandaReach-v3", render_mode="rgb_array", render_width=480, render_height=360,
                                   render_target_position=[0.0, 0.0, 0.1], render_distance=1.1,
                                   render_yaw=50, render_pitch=-25))
frames = []
for i in range(5):
    obs, info = venv.reset(seed=2_000_000 + i)
    frames.append(venv.render())
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = venv.step(action)
        done = terminated or truncated
        frames.append(venv.render())
    frames += [frames[-1]] * 6
imageio.mimsave("panda_reach.mp4", frames, fps=25, macro_block_size=8)
print(len(frames), "frames -> panda_reach.mp4")

In [ ]:
from IPython.display import Video
Video("panda_reach.mp4", embed=True, width=480)